In [1]:
!pip install langchain-community langchain-google-genai faiss-cpu tiktoken youtube-transcript-api python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from google.colab import userdata


llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=userdata.get("GOOGLE_API_KEY"),
)

/tmp/ipykernel_2915/220981638.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


 Document Ingestion

In [3]:
video_id='67_aMPDk2zw'

try:
  transcript_list=YouTubeTranscriptApi().fetch(video_id,languages=['en']) #extracting captions wdith timestamps
  transcript=" ".join(snippet.text for snippet in transcript_list) #just gives a simple plain joined text of transcript lists
  # for snippet in transcript_list:
  #       print(f"Start:    {snippet.start} seconds")
  #       print(f"Duration: {snippet.duration} seconds")
  #       print(f"Text:     {snippet.text}")
  #       print("-" * 60)
  # print(transcript)
except Exception as e :
  # print("No caption available for this video")
  print(f"error: {e}")

Text Splitting

In [4]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.create_documents([transcript])

Embedding Generation & Storing in vector space

In [5]:
embeddings=GoogleGenerativeAIEmbeddings(google_api_key=userdata.get("GOOGLE_API_KEY"),model="gemini-embedding-2-preview")
vectorstore=FAISS.from_documents(chunks,embeddings)


Retrieval

In [6]:
retriever=vectorstore.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [7]:
retriever.invoke("what has been taught in this video")

[Document(id='7fd404ac-ae31-4827-b81c-17b7a7608847', metadata={}, page_content="learning with human feedback rlhf let's understand this once again with Buddy one day Peter was having a conversation with his cute little two-year-old son don't eat too much bananas else hearing this Peter realized that buddy has been listening to the conversations from abusive parents in his town what he said was the effect of that Peter then starts skipping a close eye on what buddy is saying for a same question buddy can produce multiple answers and all Peter has to do is tell him which one is toxic and which one is not after this training buddy doesn't use any toxic language while training chat GPT open air used a similar approach of human intervention rlhf open air used a huge Workforce of humans to make chat GPT less toxic while llms are very powerful they don't have any subjective experience emotions or Consciousness that we as humans have llms work purely based on the data that they have been train

Augmentation

In [9]:
prompt=PromptTemplate(
    template="""
    You are a helpful assistant
    answer from the provided transcript context
    you can use your thinking also to give answers
    {context}
    Question: {question}
    """,
    input_variables=["context","question"]
)

question="Is this video LLm related"
retrieved_docs=retriever.invoke(question)

In [10]:
context_text="/n/n".join(doc.page_content for doc in retrieved_docs)
final_prompt=prompt.invoke({"context":context_text,"question":question})

In [11]:
final_prompt

StringPromptValue(text="\n    You are a helpful assistant\n    answer from the provided transcript context\n    you can use your thinking also to give answers\n    experience emotions or Consciousness that we as humans have llms work purely based on the data that they have been trained on I hope you like this short explanation which was based on analogy obviously the technical working of this thing is little different than analogy but this should give you a good intuition on this topic if you like this video please share with those who are curious about this topic foreign/n/nlearning with human feedback rlhf let's understand this once again with Buddy one day Peter was having a conversation with his cute little two-year-old son don't eat too much bananas else hearing this Peter realized that buddy has been listening to the conversations from abusive parents in his town what he said was the effect of that Peter then starts skipping a close eye on what buddy is saying for a same question

Generation

In [12]:
answer=llm.invoke(final_prompt)
print(answer.content[0]["text"])

**Yes, this video is directly related to Large Language Models (LLMs).** 

The transcript uses an analogy (a mimicking parrot named Buddy as a "stochastic parrot") to explain:
* How **LLMs** work by predicting the next set of words using statistical probability and neural networks.
* How LLMs are trained on vast amounts of data (such as Wikipedia, books, and news articles).
* Specific examples of LLMs and related tools, such as **ChatGPT (GPT-3/GPT-4)**, Google's **PaLM 2**, and Meta's **LLaMA**.
* Concepts like **RLHF** (Reinforcement Learning with Human Feedback) to make LLMs less toxic.
* Clarifications that LLMs do not have human consciousness or emotions, but operate based purely on training data.


Chaining

In [13]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def format_docs(retrieved_docs):
  context_text="/n/n".join(doc.page_content for doc in retrieved_docs)
  return context_text


In [14]:
parallel_chain=RunnableParallel({
    'context':retriever | RunnableLambda(format_docs),
    'question':RunnablePassthrough()
})

In [15]:
parallel_chain.invoke("who is pal?")

{'context': "parrot their computer programs that use a technology called neural networks to predict the next set of words for a sentence for a simple explanation of a neural network please watch this particular video just like how birdies strain on Peter's home conversations data set you can have a language model that is trained on for example all movie related articles from Wikipedia and it will be able to predict the next set of words for a movie related sentence Gmail autocomplete is one of the many applications that uses a language model underneath now that we have some understanding of a language model let's understand what the heck is a large language model let's go back to our buddy example our buddy got some Divine super power and now he can listen to Peter's neighbors conversations conversations that are happening in schools and universities in the town in fact not only in his town but all the towns across the world with this extra power and knowledge now buddy can complete th

In [16]:
parser=StrOutputParser()

In [17]:
main_chain=parallel_chain | prompt|llm|parser

In [18]:
main_chain.invoke("Is this video explained how llm works?")

'**Yes, the video explains how Large Language Models (LLMs) work**, primarily through an analogy of a parrot named "Buddy" (a stochastic parrot). \n\nHere is how the video breaks it down:\n1. **Next-Word Prediction:** It explains that language models use neural networks and statistical probability (along with some randomness) to predict the next set of words based on training data, much like the parrot mimics conversations it has heard without truly understanding their meaning.\n2. **From Language Models to LLMs:** A standard language model is compared to a parrot trained on a small dataset (like conversations in one house), whereas an LLM is compared to a parrot with the "superpower" of listening to conversations across the entire world, allowing it to complete sentences on a massive variety of topics.\n3. **Training & RLHF:** It also explains how human feedback (RLHF—Reinforcement Learning with Human Feedback) is used to fine-tune LLMs (like OpenAI\'s ChatGPT) to make them less toxic